# Full Multi-Modal RAG Pipeline [Step 4 - Query, Retrieve, Generate with Vision]

> **MLCourse - Agentic AI - Multi-Modal RAG**

### What you will learn

1. How to build a complete multi-modal RAG pipeline with LangGraph.
2. Query routing: decide whether to search text, images, or both.
3. Retrieval: pull relevant text chunks and images from ChromaDB.
4. Generation: feed retrieved context (text + images) to OpenAI gpt-4o
   for multi-modal answer generation.
5. Guard patterns: how the pipeline works when OpenAI is unavailable.

This notebook ties together everything from notebooks 01-03 into a
production-style RAG pipeline. The user asks a question, the system
retrieves both text and images, and an LLM generates an answer using
all modalities.

In [1]:
# ---------------------------------------------------------------------------
# Setup cell (identical in every MLCourse notebook): imports, TRACK walker,
# DATA folder creation, .env loading, matplotlib inline magic - guarded so
# the file also runs as a plain script outside Jupyter.
# ---------------------------------------------------------------------------
from pathlib import Path


def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until a directory named ``target`` shows up."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(
        f"Could not find '{target}' above {start}. "
        "Open this notebook from inside the MLCourse repository."
    )


TRACK = find_track(Path.cwd())       # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"                # one shared data folder for the track
DATA.mkdir(exist_ok=True)            # no-op when it already exists

from dotenv import load_dotenv       # noqa: E402  reads KEY=value files

load_dotenv()                        # .env beside the current directory
load_dotenv(TRACK / ".env")          # .env at the track root

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                             # silently skip the magic outside IPython

import matplotlib.pyplot as plt      # noqa: E402

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)

[setup] TRACK: D:\projects\python\MLCourse\03_agentic_ai
[setup] DATA : D:\projects\python\MLCourse\03_agentic_ai\data


### Part 1: Install Dependencies and Load Models


In [ ]:
import subprocess, sys, os

def install_if_missing(package: str, import_name: str = None):
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("open-clip-torch", "open_clip")
install_if_missing("chromadb")
install_if_missing("Pillow")
install_if_missing("pymupdf", "fitz")
install_if_missing("langgraph")
install_if_missing("langchain-ollama")
install_if_missing("langchain-openai")

import torch
import open_clip
from PIL import Image
import chromadb
import numpy as np
import fitz
import io
from typing import TypedDict, Annotated, Sequence
import operator


### Part 2: Load CLIP for Embedding Queries


In [ ]:
#
# We need CLIP at query time to embed the user's text query so it can
# search both the text and image collections.

device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device).eval()

print(f"[CLIP] Loaded on {device}")


### Part 3: Load LLMs


In [ ]:
#
# Two LLMs serve different roles:
#   - ChatOllama (llama3.1:8b): free, local, used for text-only answers
#   - OpenAI gpt-4o: used ONLY for multi-modal generation (text + images)

from langchain_ollama import ChatOllama

LOCAL_LLM = ChatOllama(model="llama3.1:8b", temperature=0)

OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "")
if OPENAI_KEY:
    from langchain_openai import ChatOpenAI
    VISION_LLM = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=1024)
    print("[LLM] ChatOllama (llama3.1:8b) + OpenAI (gpt-4o) ready")
else:
    VISION_LLM = None
    print("[LLM] ChatOllama (llama3.1:8b) ready -- no OpenAI key, vision generation guarded")


### Part 4: Build or Load the Multi-Modal Index


In [ ]:
#
# We ensure the ChromaDB index exists before querying it.

CHROMA_DIR = TRACK / "data" / "chroma_multimodal"
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

def build_index():
    """Extract and index the PDF from scratch."""
    print("[INDEX] Building index...")
    PDF_PATH = DATA / "attention_is_all_you_need.pdf"
    EXTRACT_DIR = DATA / "multimodal_images"
    EXTRACT_DIR.mkdir(exist_ok=True)

    doc = fitz.open(str(PDF_PATH))

    extracted_images = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        for img_idx, img_info in enumerate(page.get_images(full=True)):
            xref = img_info[0]
            base_image = doc.extract_image(xref)
            img_bytes = base_image["image"]
            ext = base_image["ext"]
            img_name = f"page{page_num + 1}_img{img_idx + 1}.{ext}"
            img_path = EXTRACT_DIR / img_name
            with open(img_path, "wb") as f:
                f.write(img_bytes)
            img = Image.open(io.BytesIO(img_bytes))
            w, h = img.size
            if w < 50 or h < 50:
                img_path.unlink()
                continue
            extracted_images.append({
                "path": str(img_path), "name": img_name,
                "page": page_num + 1, "width": w, "height": h,
            })

    text_chunks = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        for para_idx, para in enumerate(paragraphs):
            if len(para) < 20:
                continue
            text_chunks.append({
                "text": para, "page": page_num + 1,
                "chunk_id": f"page{page_num + 1}_para{para_idx}",
                "source": PDF_PATH.name,
            })
    doc.close()

    # Embed texts.
    all_text = [c["text"] for c in text_chunks]
    text_vecs = []
    for i in range(0, len(all_text), 32):
        batch = all_text[i:i + 32]
        tokens = tokenizer(batch).to(device)
        with torch.no_grad():
            vecs = model.encode_text(tokens)
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
        text_vecs.append(vecs.cpu())
    text_emb = torch.cat(text_vecs, dim=0)

    # Embed images.
    img_tensors = [preprocess(Image.open(p).convert("RGB")).unsqueeze(0)
                   for p in [e["path"] for e in extracted_images]]
    img_vecs = []
    for i in range(0, len(img_tensors), 8):
        batch = torch.cat(img_tensors[i:i + 8], dim=0).to(device)
        with torch.no_grad():
            vecs = model.encode_image(batch)
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
        img_vecs.append(vecs.cpu())
    img_emb = torch.cat(img_vecs, dim=0) if img_vecs else torch.zeros(0, 512)

    # Create collections.
    try:
        client.delete_collection("multimodal_text")
        client.delete_collection("multimodal_images")
    except Exception:
        pass

    tc = client.create_collection("multimodal_text", metadata={"hnsw:space": "cosine"})
    ic = client.create_collection("multimodal_images", metadata={"hnsw:space": "cosine"})

    tc.add(
        ids=[c["chunk_id"] for c in text_chunks],
        embeddings=text_emb.numpy().tolist(),
        documents=all_text,
        metadatas=[{"page": c["page"], "source": c["source"], "type": "text"}
                    for c in text_chunks],
    )
    if extracted_images:
        ic.add(
            ids=[e["name"] for e in extracted_images],
            embeddings=img_emb.numpy().tolist(),
            documents=[e["name"] for e in extracted_images],
            metadatas=[{"page": e["page"], "path": e["path"],
                         "width": e["width"], "height": e["height"], "type": "image"}
                        for e in extracted_images],
        )
    print(f"[INDEX] Built: {tc.count()} text, {ic.count()} images")
    return tc, ic

try:
    text_col = client.get_collection("multimodal_text")
    img_col = client.get_collection("multimodal_images")
    if text_col.count() == 0:
        text_col, img_col = build_index()
    else:
        print(f"[INDEX] Loaded: {text_col.count()} text, {img_col.count()} images")
except Exception:
    text_col, img_col = build_index()


### Part 5: Define the RAG State


In [ ]:
#
# The pipeline state tracks the user query, retrieved context, and
# the final answer. This is the TypedDict that flows through the graph.

class RAGState(TypedDict):
    query: str                                    # user question
    retrieved_texts: list[dict]                   # text chunks with metadata
    retrieved_images: list[dict]                  # images with metadata
    context_text: str                             # formatted text context
    answer: str                                   # final LLM answer
    query_embedding: list                         # CLIP vector for the query


### Part 6: Define the Retrieve Node


In [ ]:
#
# The retrieve node embeds the query with CLIP and searches both
# ChromaDB collections, returning text and image results.

def retrieve(state: RAGState) -> dict:
    """Embed the query and retrieve text + images from ChromaDB."""
    query = state["query"]
    print(f"\n[RETRIEVE] Processing query: \"{query}\"")

    # Embed query with CLIP.
    tokens = tokenizer([query]).to(device)
    with torch.no_grad():
        qvec = model.encode_text(tokens)
        qvec = qvec / qvec.norm(dim=-1, keepdim=True)
    qvec_np = qvec.numpy().tolist()

    # Search text collection.
    text_results = text_col.query(query_embeddings=qvec_np, n_results=5)
    retrieved_texts = []
    for i in range(len(text_results["ids"][0])):
        retrieved_texts.append({
            "id": text_results["ids"][0][i],
            "text": text_results["documents"][0][i],
            "distance": text_results["distances"][0][i],
            "metadata": text_results["metadatas"][0][i],
        })

    # Search image collection.
    retrieved_images = []
    if img_col.count() > 0:
        n_img = min(3, img_col.count())
        img_results = img_col.query(query_embeddings=qvec_np, n_results=n_img)
        for i in range(len(img_results["ids"][0])):
            retrieved_images.append({
                "id": img_results["ids"][0][i],
                "name": img_results["documents"][0][i],
                "distance": img_results["distances"][0][i],
                "metadata": img_results["metadatas"][0][i],
            })

    # Format text context for the LLM.
    context_parts = []
    for rank, t in enumerate(retrieved_texts, 1):
        sim = 1 - t["distance"]
        context_parts.append(
            f"[Source {rank}] (page {t['metadata']['page']}, relevance={sim:.3f}):\n"
            f"{t['text']}"
        )
    context_text = "\n\n".join(context_parts)

    print(f"[RETRIEVE] Found {len(retrieved_texts)} text chunks, "
          f"{len(retrieved_images)} images")

    return {
        "retrieved_texts": retrieved_texts,
        "retrieved_images": retrieved_images,
        "context_text": context_text,
        "query_embedding": qvec_np,
    }


### Part 7: Define the Generate Node (Text-Only, Local)


In [ ]:
#
# When OpenAI is unavailable or for quick answers, we use the local
# ChatOllama to generate a text-only answer from retrieved text context.

def generate_local(state: RAGState) -> dict:
    """Generate answer using local LLM and text context only."""
    prompt = (
        "You are a helpful assistant. Answer the user's question based ONLY "
        "on the provided context. If the context does not contain enough "
        "information, say so honestly. Be concise and accurate.\n\n"
        f"Context:\n{state['context_text']}\n\n"
        f"Question: {state['query']}\n\n"
        "Answer:"
    )

    response = LOCAL_LLM.invoke(prompt)
    print(f"[GENERATE-LOCAL] Answer length: {len(response.content)} chars")
    return {"answer": response.content}


### Part 8: Define the Generate Node (Multi-Modal, OpenAI)


In [ ]:
#
# When OpenAI is available, we build a multi-modal message that includes
# both text context AND images, then send it to gpt-4o for a richer answer.

import base64

def generate_multimodal(state: RAGState) -> dict:
    """Generate answer using OpenAI vision LLM with text + images."""
    if VISION_LLM is None:
        return generate_local(state)

    from langchain_core.messages import HumanMessage

    # Build the message content list.
    content = []

    # Text context.
    content.append({
        "type": "text",
        "text": (
            "You are a helpful assistant. Answer the question based on the "
            "following context (text and images from a research paper). "
            "Reference both text and visual information in your answer.\n\n"
            f"Text context:\n{state['context_text']}\n\n"
            f"Question: {state['query']}"
        ),
    })

    # Add retrieved images as base64.
    for img_info in state["retrieved_images"]:
        img_path = img_info["metadata"].get("path", "")
        if img_path and Path(img_path).exists():
            with open(img_path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{b64}",
                    "detail": "low",   # save tokens, enough for diagrams
                },
            })

    message = HumanMessage(content=content)
    response = VISION_LLM.invoke([message])

    print(f"[GENERATE-VISION] Answer length: {len(response.content)} chars "
          f"(with {len(state['retrieved_images'])} images)")
    return {"answer": response.content}


### Part 9: Build the LangGraph Pipeline


In [ ]:
#
# The graph has three nodes:
#   1. retrieve: embed query, search ChromaDB
#   2. generate: produce answer (local or multi-modal)
#
# We use a simple linear flow: START -> retrieve -> generate -> END.

from langgraph.graph import StateGraph, START, END

graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate_local", generate_local)
graph.add_node("generate_multimodal", generate_multimodal)

graph.add_edge(START, "retrieve")

# Conditional edge: use vision LLM if available, else local.
def route_generation(state: RAGState) -> str:
    if VISION_LLM is not None and state.get("retrieved_images"):
        return "generate_multimodal"
    return "generate_local"

graph.add_conditional_edges("retrieve", route_generation, {
    "generate_multimodal": "generate_multimodal",
    "generate_local": "generate_local",
})

graph.add_edge("generate_local", END)
graph.add_edge("generate_multimodal", END)

app = graph.compile()


### Part 10: Visualize the Pipeline


In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph visualization unavailable in this environment.")
    print("Pipeline: START -> retrieve -> [generate_local | generate_multimodal] -> END")


### Part 11: Run the Pipeline


In [ ]:
#
# Let us test the full pipeline with several queries about the
# transformer paper.

def run_rag(query: str) -> dict:
    """Run the full multi-modal RAG pipeline on a query."""
    print("=" * 70)
    result = app.invoke({
        "query": query,
        "retrieved_texts": [],
        "retrieved_images": [],
        "context_text": "",
        "answer": "",
        "query_embedding": [],
    })
    print(f"\n[ANSWER]\n{result['answer']}")
    return result

# Query 1: Technical question about attention.
result1 = run_rag("What is multi-head attention and why is it used?")


### Query 2: Question that likely involves diagrams.


In [ ]:
result2 = run_rag("Describe the overall architecture of the transformer model")


### Query 3: Specific factual question.


In [ ]:
result3 = run_rag("What training hardware was used for the transformer model?")


### Part 12: Show Retrieved Context and Images


In [ ]:
#
# For debugging and understanding, let us see exactly what was
# retrieved for each query.

debug_queries = [
    "self-attention mechanism",
    "encoder decoder architecture",
]

for q in debug_queries:
    print(f"\n{'=' * 70}")
    print(f"Query: \"{q}\"")
    print("=" * 70)

    # Retrieve without generating (to show raw results).
    tokens = tokenizer([q]).to(device)
    with torch.no_grad():
        qvec = model.encode_text(tokens)
        qvec = qvec / qvec.norm(dim=-1, keepdim=True)
    qvec_np = qvec.numpy().tolist()

    text_res = text_col.query(query_embeddings=qvec_np, n_results=3)
    print("\nRetrieved text chunks:")
    for i in range(len(text_res["ids"][0])):
        sim = 1 - text_res["distances"][0][i]
        page = text_res["metadatas"][0][i]["page"]
        doc = text_res["documents"][0][i]
        print(f"  #{i+1} sim={sim:.4f} page={page}")
        print(f"      {doc[:120]}...")

    if img_col.count() > 0:
        img_res = img_col.query(query_embeddings=qvec_np, n_results=2)
        print("\nRetrieved images:")
        for i in range(len(img_res["ids"][0])):
            sim = 1 - img_res["distances"][0][i]
            meta = img_res["metadatas"][0][i]
            print(f"  #{i+1} sim={sim:.4f} page={meta['page']} name={img_res['documents'][0][i]}")


### Part 13: Pipeline Architecture Summary


In [ ]:
print("\n" + "=" * 70)
print("MULTI-MODAL RAG PIPELINE ARCHITECTURE")
print("=" * 70)
print()
print("  User Query")
print("       |")
print("       v")
print("  +------------+")
print("  | Embed with |  (CLIP ViT-B-32, shared 512-dim space)")
print("  | CLIP       |")
print("  +------+-----+")
print("       |")
print("       +-----> ChromaDB 'multimodal_text'  (top 5 text chunks)")
print("       |")
print("       +-----> ChromaDB 'multimodal_images' (top 3 images)")
print("       |")
print("       v")
print("  +--------------------+")
print("  | Route: OpenAI key? |")
print("  +---+------------+---+")
print("      |            |")
print("      v (yes)      v (no)")
print("  +--------+   +-----------+")
print("  | gpt-4o |   | llama3.1  |")
print("  | text + |   | text only |")
print("  | images |   +-----------+")
print("  +----+---+       |")
print("       |           |")
print("       v           v")
print("  +-------------------+")
print("  |   Final Answer    |")
print("  +-------------------+")


### Part 14: Cost and Performance Notes


In [ ]:
print("\nCost breakdown per query (approximate):")
print("-" * 50)
print("  CLIP embedding (query):   free (local)")
print("  ChromaDB search:          free (local)")
print("  Text generation (local):  free (ChatOllama, llama3.1:8b)")
print("  Vision generation:        ~$0.01-0.03 per query (gpt-4o, low detail)")
print()
print("Performance notes:")
print("-" * 50)
print("  CLIP query embedding:     <50ms")
print("  ChromaDB search:          <10ms")
print("  Local LLM generation:     2-10 seconds")
print("  OpenAI vision generation: 2-5 seconds")
print()
print("Guard patterns used:")
print("-" * 50)
print("  - No OpenAI key -> falls back to local text-only LLM")
print("  - No images retrieved -> falls back to text-only generation")
print("  - Empty index -> index is rebuilt from PDF automatically")


### Summary


In [ ]:
#
# 1. The pipeline: query -> CLIP embed -> ChromaDB search -> LLM generate.
# 2. Query routing decides between local and multi-modal generation.
# 3. OpenAI gpt-4o handles text + images; ChatOllama handles text only.
# 4. The guard pattern ensures the pipeline works without an API key.
# 5. All components are modular and can be swapped independently.
#
# This concludes Module 21: Multi-Modal RAG. You now have a complete
# understanding of:
#   - CLIP shared embedding spaces (notebook 01)
#   - PDF image extraction and indexing (notebook 02)
#   - Cross-modal retrieval (notebook 03)
#   - Full multi-modal RAG pipeline (notebook 04)

print("\n[COMPLETE] Module 21 Notebook 4: Multimodal RAG Pipeline")
print("  - LangGraph pipeline with retrieve + generate nodes")
print("  - Query routing between local and vision LLM")
print("  - Text and image retrieval from ChromaDB")
print("  - Multi-modal generation with gpt-4o (guarded)")
print("  - Graceful fallback to local generation without API key")
